<a href="https://colab.research.google.com/github/gauravd12345/computer_vision/blob/main/vision_transformer/vision_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001-1359597a978bc4fa.parquet', 'valid': 'data/valid-00000-of-00001-70d52db3c749a935.parquet'}
df = pd.read_parquet("hf://datasets/zh-plus/tiny-imagenet/" + splits["train"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
""" imports """

import torch
import torch.nn as nn

import io
from PIL import Image
import matplotlib.pyplot as plt

images, labels = df['image'], df['label']

In [7]:
""" hyperparameters """
N_images = len(images)
H = 64                        # resolution of images
W = 64
C = 3                         # number of channels (r, g, b)

P = 8                         # resolution of image patches
N = (H * W) // (P**2)         # number of image patches

D = 128                       # embedding dim

print(f"Number of images: {N_images}")
print(f"H: {H}, W: {W}, C: {C}")
print(f"P: {P}, N: {N}")

Number of images: 100000
H: 64, W: 64, C: 3
P: 8, N: 64


In [4]:
from torchvision import transforms

transform = transforms.ToTensor()  # converts PIL image to (C, H, W) tensor, scales to [0, 1]

X_train, y_train = [], []
for i, l in zip(images, labels):
    img = Image.open(io.BytesIO(i['bytes'])).convert('RGB')
    X_train.append(transform(img))
    y_train.append(l)

X_train = torch.stack(X_train).permute(0, 2, 3, 1)
y_train = torch.tensor(y_train)

In [5]:
print(X_train.shape, y_train.shape)

torch.Size([100000, 64, 64, 3]) torch.Size([100000])


In [ ]:
""" flattening images """

X_train = X_train.reshape(N_images, N, P**2 * C)
print(X_train.shape)